In [ ]:
import os
import numpy as np
import pickle
from pathlib import Path
from openwakeword.model import Model
import librosa

POSITIVE_DIR = Path("dataphase1/positive")
NEGATIVE_DIR = Path("dataphase1/negative")
MODEL_OUT    = Path("models")
MODEL_OUT.mkdir(exist_ok=True)

def extract_embedding(wav_path):
    """Dùng OpenWakeWord pre-trained model để extract embeddings."""
    audio, _ = librosa.load(str(wav_path), sr=16000, mono=True)
    audio_int16 = (audio * 32767).astype(np.int16)
    oww = Model(inference_framework="onnx")
    oww.predict(audio_int16)
    # Lấy embedding từ layer cuối
    emb = oww.preprocessor.get_features(audio_int16)
    return np.mean(emb, axis=0) if emb is not None else None

print("Loading data...")
X, y = [], []

for wav in POSITIVE_DIR.glob("*.wav"):
    try:
        audio, _ = librosa.load(str(wav), sr=16000, mono=True)
        mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=40)
        X.append(np.mean(mfcc.T, axis=0))
        y.append(1)
    except: pass

for wav in NEGATIVE_DIR.glob("*.wav"):
    try:
        audio, _ = librosa.load(str(wav), sr=16000, mono=True)
        mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=40)
        X.append(np.mean(mfcc.T, axis=0))
        y.append(0)
    except: pass

X, y = np.array(X), np.array(y)
print(f"Dataset: {len(X)} samples (pos={y.sum()}, neg={(1-y).sum()})")

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='f1')
print(f"Cross-val F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

clf.fit(X_scaled, y)

pickle.dump(clf, open(MODEL_OUT / "model.pkl", "wb"))
pickle.dump(scaler, open(MODEL_OUT / "scaler.pkl", "wb"))
print(f"Model saved to models/")

Loading data...
Dataset: 240 samples (pos=140, neg=100)
Cross-val F1: 0.9896 (+/- 0.0138)
Model saved to models/
